# Instant Personality Steering with PSYCTL

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/modulabs-personalab/psyctl/blob/main/examples/en/01_quickstart.ipynb)

**What is PSYCTL?** A Python toolkit for steering LLM personalities using [Contrastive Activation Addition (CAA)](https://arxiv.org/abs/2312.06681) and [Bidirectional Preference Optimization (BiPO)](https://arxiv.org/abs/2406.00045). Unlike prompt engineering, PSYCTL modifies model activations directly — making personality changes consistent and measurable.

**In this notebook you will:**
1. Load a pre-trained BiPO steering vector and see instant personality changes
2. Compare baseline vs steered responses at different strengths
3. Explore all 5 available personality vectors side-by-side

**Requirements:** Colab free tier (T4 GPU) is sufficient. You need a [HuggingFace token](https://huggingface.co/settings/tokens) with access to [Llama-3.1-8B-Instruct](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct).

**Time:** ~5 minutes

In [ ]:
!pip install -q git+https://github.com/modulabs-personalab/psyctl.git bitsandbytes accelerate

In [ ]:
import os

try:
    from google.colab import userdata
    _hf = userdata.get("HF_TOKEN")
    os.environ["HF_TOKEN"] = _hf if isinstance(_hf, str) else str(_hf)
    print("Loaded HF_TOKEN from Colab Secrets")
except Exception:
    if not os.environ.get("HF_TOKEN"):
        raise RuntimeError(
            "HF_TOKEN not found. Please:\n"
            "1. Click the key icon (Secrets) in the left sidebar\n"
            "2. Add HF_TOKEN with your HuggingFace token\n"
            "3. Toggle 'Notebook access' ON for HF_TOKEN\n"
            "4. Re-run this cell"
        )

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PSYCTL_LOG_LEVEL"] = "WARNING"

## Setup: Load Model

We load Llama-3.1-8B-Instruct in 4-bit quantization (~5 GB VRAM) so it fits on a free Colab T4.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

print(f"Loading {MODEL_ID} in 4-bit...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded on {model.device} | Memory: {model.get_memory_footprint() / 1e9:.1f} GB")

## Download a Steering Vector

We download a pre-trained **Agreeableness** BiPO vector from [dalekwon/bipo-steering-vectors](https://huggingface.co/dalekwon/bipo-steering-vectors).

In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path

VECTOR_REPO = "dalekwon/bipo-steering-vectors"
VECTOR_DIR = Path("./vectors")
VECTOR_DIR.mkdir(exist_ok=True)

vector_path = Path(hf_hub_download(
    repo_id=VECTOR_REPO,
    filename="bipo_steering_english_agreeableness.safetensors",
    local_dir=str(VECTOR_DIR),
    token=os.environ.get("HF_TOKEN"),
))
print(f"Downloaded vector: {vector_path}")

## Baseline vs Steered

The same prompt produces dramatically different responses depending on the steering strength. Positive strength increases agreeableness; negative strength decreases it.

In [ ]:
from psyctl.core.steering_applier import SteeringApplier

applier = SteeringApplier()


def generate_baseline(prompt: str, max_new_tokens: int = 200) -> str:
    """Generate without steering."""
    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            temperature=0.7, top_p=0.9, do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(output_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)


def generate_steered(prompt: str, strength: float, vpath=vector_path, max_new_tokens: int = 200) -> str:
    """Generate with steering vector applied."""
    return applier.apply_steering(
        model=model,
        tokenizer=tokenizer,
        steering_vector_path=vpath,
        input_text=prompt,
        strength=strength,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        top_p=0.9,
    )

In [ ]:
prompt = "My coworker keeps taking credit for my ideas. What should I do?"

print(f"Prompt: {prompt}")
print("=" * 70)

print("\n--- Baseline (no steering) ---")
print(generate_baseline(prompt))

print("\n--- Steered: Agreeableness +2.0 ---")
print(generate_steered(prompt, strength=2.0))

print("\n--- Steered: Agreeableness -2.0 (disagreeable) ---")
print(generate_steered(prompt, strength=-2.0))

## Strength Sweep

See how personality changes gradually from -2.0 to +3.0.

In [ ]:
prompt = "Someone cuts in line at the coffee shop. How do you react?"
strengths = [-2.0, -1.0, 0.0, 1.0, 2.0, 3.0]

print(f"Prompt: {prompt}")
print("=" * 70)

for s in strengths:
    label = "Baseline" if s == 0.0 else f"Agreeableness {s:+.1f}"
    print(f"\n--- {label} ---")
    if s == 0.0:
        print(generate_baseline(prompt, max_new_tokens=150))
    else:
        print(generate_steered(prompt, strength=s, max_new_tokens=150))

## Explore All Personalities

The [dalekwon/bipo-steering-vectors](https://huggingface.co/dalekwon/bipo-steering-vectors) repo contains 5 English personality vectors:

| Vector | Description |
|--------|-------------|
| `agreeableness` | Cooperative, trusting, compassionate |
| `neuroticism` | Emotionally reactive, anxious, sensitive |
| `awfully_sweet` | Extremely kind, warm, effusively positive |
| `paranoid` | Suspicious, distrustful, hypervigilant |
| `very_lascivious` | Bold, sensation-seeking, boundary-pushing |

In [ ]:
VECTOR_FILES = {
    "agreeableness": "bipo_steering_english_agreeableness.safetensors",
    "neuroticism": "bipo_steering_english_neuroticism.safetensors",
    "awfully_sweet": "bipo_steering_english_awfully_sweet.safetensors",
    "paranoid": "bipo_steering_english_paranoid.safetensors",
    "very_lascivious": "bipo_steering_english_very_lascivious.safetensors",
}

vector_paths = {}
for name, filename in VECTOR_FILES.items():
    vector_paths[name] = Path(hf_hub_download(
        repo_id=VECTOR_REPO, filename=filename,
        local_dir=str(VECTOR_DIR), token=os.environ.get("HF_TOKEN"),
    ))
    print(f"  {name}: ready")

print(f"\nAll {len(vector_paths)} vectors downloaded.")

In [ ]:
# Side-by-side comparison: same prompt, all personalities
prompt = "Tell me about yourself."
strength = 2.5

print(f"Prompt: \"{prompt}\" (strength={strength:+.1f})")
print("=" * 70)

print("\n[Baseline]")
print(generate_baseline(prompt, max_new_tokens=120))

for name, vpath in vector_paths.items():
    print(f"\n[{name}]")
    print(generate_steered(prompt, strength=strength, vpath=vpath, max_new_tokens=120))
    print("-" * 70)

## Next Steps

- **[02_measure_personality.ipynb](./02_measure_personality.ipynb)** — Quantify personality shifts with standardized psychology inventories
- **[04_extract_vector.ipynb](./04_extract_vector.ipynb)** — Extract your own custom steering vectors
- **[GitHub](https://github.com/modulabs-personalab/psyctl)** — Star the repo, open issues, contribute